# 05b - (Tahap 2) Cleaning dataset QA

In [1]:
!pip install -q sentence-transformers faiss-cpu huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 30.2 MB/s eta 0:00:00


In [2]:
import json, os, re, datetime
import numpy as np
import faiss
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient
import torch
import random

## 1. Konfigurasi & Load

In [3]:
CONFIG = {
    "input_dir":               "/kaggle/input/notebooks/muhammadnabilul/05-generating-context-awareqa/output_05a_v2",
    "output_dir":              "output_05b",
    "hf_repo_id":              "Makaareeem/publikasi-rag-knowledge-base",
    "embedding_model":         "nomic-ai/nomic-embed-text-v1.5",
    "dedup_threshold":         0.97,
    "dedup_max_per_cluster":   2,
    "dedup_max_per_cluster_lenient": 6,
    "dedup_lenient_subtasks":  {"definition_clarification", "methodology_explanation"},
    "degenerate_min_words":            6,
    "degenerate_min_words_by_subtask": {
        "source_navigation": 3,
        "factual_retrieval": 3,
    },
    "chunk_id_hard_fail_ratio": 0.5,
    "embed_batch_size":        128,
    "preview_n":               4,
    "preview_seed":            42,
    "dedup_seed":              42,
    "cot_min_span_overlap":    0.5,
}
 
ALLOWED_TASKS = {"qa", "summary"}
ALLOWED_SUBTASKS = {
    "factual_retrieval", "source_navigation",
    "definition_clarification", "methodology_explanation",
    "trend_interpretation", "causal_reasoning",
    "comparative_analysis", "multi_source_synthesis",
    "document_overview",
}

DEFINITION_SUBTASKS = {"definition_clarification", "methodology_explanation"}
 
TABLE_CELL_READ_PATTERNS = [
    r"berapa (nilai|angka) (pada|di) (baris|kolom|sel)",
    r"pada baris (ke-?\d+|nomor \d+)",
    r"nilai (pada|di) kolom",
]
 
TABLE_REF_PHRASE_PATTERNS = [
    r"dalam tabel ini",
    r"dalam konteks tabel ini",
    r"dalam tabel berikut",
    r"tabel di atas",
    r"dalam tabel tersebut",
]
 
DEFINITION_QUESTION_PATTERNS = [
    r"^apa yang dimaksud dengan",
    r"^apa (itu|arti|definisi)",
    r"^bagaimana (metode|cara|prosedur).*dijelaskan",
    r"^jelaskan (istilah|konsep|metode)",
]
 
META_REF_AWALAN = [
    r"^(berdasarkan|menurut|sesuai|dalam)\s+(teks sumber|passages|konteks|dokumen ini)\b[,.]?\s*",
    r"^(teks sumber|passages|konteks|dokumen ini)\s+(ini\s+)?menunjukkan\b[,.]?\s*",
    r"^(teks sumber|passages|konteks|dokumen ini)\s+(ini\s+)?menyatakan\b[,.]?\s*",
    r"^(teks sumber|passages|konteks|dokumen ini)\s+(ini\s+)?menjelaskan\b[,.]?\s*",
]
 
LEAK_META_PATTERNS = [
    r"\b(konteks|passages)\s+(ini|yang diberikan|yang tersedia)\b",
    r"berdasarkan\s+(teks sumber|passages|konteks)\s+(ini|yang diberikan)\b",
    r"\bteks sumber\s+(ini|yang diberikan)\b",
    r"\(tidak disebut eksplisit",
    r"\(dihitung dari",
]
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
def load_raw_records(input_dir):
    path = os.path.join(input_dir, "raw_records.jsonl")
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records
 
raw_records = load_raw_records(CONFIG["input_dir"])
n_missing_spans = sum(1 for r in raw_records if not isinstance(r.get("context"), list))
print(f"Loaded {len(raw_records)} raw records")
print(f"Record dengan field 'context' BUKAN array (skema lama tercampur?): {n_missing_spans}")

Loaded 12261 raw records
Record dengan field 'context' BUKAN array (skema lama tercampur?): 0


## 2. Rule Based filtering

### a. Bersihkan teks

In [5]:
ARTIFACT_PATTERN = re.compile(r"[\xa0\u200b\u200c\u200d\ufeff]|<[^>]+>")
 
def clean_whitespace(text):
    if not text:
        return text
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()
 
n_artifact = 0
for r in raw_records:
    had = ARTIFACT_PATTERN.search(r.get("instruction","") + r.get("response",""))
    r["instruction"] = clean_whitespace(r.get("instruction",""))
    r["response"]    = clean_whitespace(r.get("response",""))
    if had:
        n_artifact += 1
 
print(f"Artefak unicode/HTML dibersihkan: {n_artifact}")

Artefak unicode/HTML dibersihkan: 3


### b. Cleaning

In [6]:
def strip_meta_awalan(text):
    """
    Hapus frasa awalan meta-reference ('berdasarkan teks sumber ini, ...')
    dari awal response, tanpa memakan kata substantif yang mengikutinya.
    Diperbaiki: versi lama memakai satu regex tunggal dengan .strip() yang
    berjalan meski pola sebenarnya tidak match, sehingga risiko memotong
    lebih dari yang seharusnya pada rangkaian pola berikutnya. Sekarang
    berhenti begitu satu pola cocok (count=1), dan hanya lstrip spasi sisa.
    """
    original = text
    for pat in META_REF_AWALAN:
        new_text = re.sub(pat, "", text, count=1, flags=re.IGNORECASE)
        if new_text != text:
            text = new_text.lstrip()
            break
    if text and text != original:
        text = text[0].upper() + text[1:]
    return text, text != original

n_cleaned = 0
for r in raw_records:
    r["flags"] = {}
    new_resp, changed = strip_meta_awalan(r.get("response",""))
    if changed:
        r["response"] = new_resp
        r["flags"]["meta_ref_cleaned"] = True
        n_cleaned += 1

print(f"Meta-reference awalan di-strip: {n_cleaned} records")

Meta-reference awalan di-strip: 14 records


### c.Valid chunks dari HF

In [7]:
secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_Faiss"))
meta_path = hf_hub_download(
    repo_id=CONFIG["hf_repo_id"],
    filename="chunk_metadata.jsonl",
    repo_type="dataset"
)
with open(meta_path, encoding="utf-8") as f:
    hf_meta = [json.loads(line) for line in f]
VALID_CHUNK_IDS = set(c["chunk_id"] for c in hf_meta)
print(f"VALID_CHUNK_IDS: {len(VALID_CHUNK_IDS)}")

chunk_metadata.jsonl:   0%|          | 0.00/45.7M [00:00<?, ?B/s]

VALID_CHUNK_IDS: 22550


### d. Batas Utama

In [8]:
def check_schema(r):
    if r.get("task_type") not in ALLOWED_TASKS:
        return False, f"task_type invalid: {r.get('task_type')}"
    if r.get("subtask_type") not in ALLOWED_SUBTASKS:
        return False, f"subtask_type invalid: {r.get('subtask_type')}"
    return True, None
 
def check_chunk_ids(r):
    ids = r.get("context_chunk_ids", [])
    if not ids:
        return False, "context_chunk_ids kosong", []
    missing = [cid for cid in ids if cid not in VALID_CHUNK_IDS]
    if not missing:
        return True, None, []
    ratio = len(missing) / len(ids)
    if ratio >= CONFIG["chunk_id_hard_fail_ratio"]:
        return False, missing, missing
    return True, None, missing
 
def check_degenerate(r):
    resp = (r.get("response") or "").strip()
    if not resp or len(resp.split()) < CONFIG["degenerate_min_words"]:
        return False, "response kosong/terlalu pendek"
    return True, None
 
def check_table_ref(r):
    instr = (r.get("instruction") or "").lower()
    for pat in TABLE_REF_PATTERNS:
        if re.search(pat, instr, re.IGNORECASE):
            return False, f"instruction merujuk tabel langsung: '{pat}'"
    return True, None

### e. Fungsi-Fungsi Utama

In [9]:
MIN_WORDS_FOR_LANG_CHECK = 12


def check_qa_quality(instruction, response):
    """
    Deteksi sampah OCR pada instruction+response (bukan context).
    Rasio karakter di luar alfanumerik+tanda baca umum yang tinggi
    menandakan teks rusak/simbol acak, bukan bahasa wajar.
    """
    qa_text = f"{instruction} {response}".strip()
    if not qa_text:
        return False, "empty_qa"
    cleaned = re.sub(r'[a-zA-Z0-9\s\.,\-\(\)%\\/:\'\"]', '', qa_text)
    if len(qa_text) > 0 and (len(cleaned) / len(qa_text)) > 0.15:
        return False, "high_ocr_noise_qa"
    return True, None


def check_context_quality(context_joined):
    """
    Deteksi sampah OCR pada context_joined. Diperbaiki: rasio karakter
    'aneh' versi lama (ambang 12%) tidak mengecualikan tanda hubung
    en/em-dash dan kutip pintar Unicode yang wajar muncul di teks BPS
    hasil ekstraksi PDF (bukan tanda kerusakan). Ambang diturunkan ke 5%
    karena sekarang lebih presisi: hanya menghitung karakter yang benar-
    benar di luar huruf, angka, dan tanda baca umum termasuk dash/kutip
    Unicode -- sisa yang tergolong 'aneh' sungguhan (mojibake residual,
    simbol kotak) menjadi proporsional lebih kecil dari sebelumnya.
    """
    if not isinstance(context_joined, str) or len(context_joined.strip()) == 0:
        return False, "empty_context"
    cleaned = re.sub(
        r'[a-zA-Z0-9\s\.,\-\(\)%\\/:\'\"\u2013\u2014\u2018\u2019\u201c\u201d]',
        '', context_joined,
    )
    ratio = len(cleaned) / len(context_joined)
    if ratio > 0.1:
        return False, "high_ocr_noise"
    return True, None


def check_mixed_english(instruction, response):
    """
    Deteksi record yang SELURUH interaksinya (instruction+response) dalam
    Bahasa Inggris -- indikasi berasal dari section dwibahasa BPS (mis.
    versi Preface berbahasa Inggris). Diperbaiki: versi lama mengukur
    rasio stopword Inggris pada context_joined, sehingga istilah teknis
    Indonesia yang kebetulan memuat substring mirip stopword Inggris
    (mis. kata 'golongan', 'internet') ikut ter-flag padahal record itu
    100% Bahasa Indonesia. Sekarang cek instruction+response secara
    langsung -- context boleh memuat sedikit istilah asing tanpa masalah.
    """
    english_stopwords = {
        "the", "and", "of", "in", "to", "for", "is", "on", "that", "by",
        "with", "as", "are", "this", "was", "were", "what", "how", "role",
    }
    text = f"{instruction} {response}".lower()
    all_words = re.findall(r"[a-zA-Z']+", text)
    if len(all_words) < MIN_WORDS_FOR_LANG_CHECK:
        return True, None
    eng_hits = [w for w in all_words if w in english_stopwords]
    eng_ratio = len(eng_hits) / len(all_words)
    if eng_ratio > 0.15:
        return False, "mixed_english_detected"
    return True, None


def check_cot_grounding(r):
    """
    Verifikasi kutipan literal (bertanda kutip) di dalam cot benar-benar
    muncul verbatim di context_joined. Diperbaiki: versi lama gagal pada
    kutipan yang memuat elipsis '...' (LLM memendekkan kutipan panjang,
    mis. 'Gambar 4.12 ... Berdasarkan Total Pendapatan') karena teks '...'
    itu literal tidak pernah ada di sumber asli. Sekarang kutipan yang
    memuat '...' dipecah jadi potongan-potongan di sekitarnya, dan tiap
    potongan diverifikasi terpisah -- tidak harus berurutan-contiguous.
    """
    cot = (r.get("cot") or "").strip()
    context_joined = r.get("context_joined", "")
    if not cot:
        return False, "cot_empty"
    quote_match = re.findall(r'"([^"]{6,})"', cot)
    if not quote_match:
        return True, None

    ctx_norm = re.sub(r"\s+", " ", context_joined).strip().lower()

    for q in quote_match:
        if "..." in q:
            parts = [p.strip() for p in q.split("...") if p.strip()]
            if not parts:
                continue
            for p in parts:
                p_norm = re.sub(r"\s+", " ", p).strip().lower()
                if len(p_norm) >= 4 and p_norm not in ctx_norm:
                    return False, f"cot_quote_not_in_context: '{p[:60]}'"
        else:
            q_norm = re.sub(r"\s+", " ", q).strip().lower()
            if q_norm not in ctx_norm:
                return False, f"cot_quote_not_in_context: '{q[:60]}'"
    return True, None


TABLE_RESIDUE_PATTERN = re.compile(
    r'\[\d+\]'
    r'|\b\d{1,3}(?:\.\d{3})*(?:,\d+)?\s+\d{1,3}(?:\.\d{3})*(?:,\d+)?\s+-?\d{1,3}(?:\.\d{3})*(?:,\d+)?\b'
)

def check_table_residue(context_spans):
    """Deteksi span yang masih memuat notasi residu tabel (footnote [n], deret label+angka tanpa penghubung)."""
    for span in context_spans:
        if TABLE_RESIDUE_PATTERN.search(span):
            return False, f"table_residue_in_span: '{span[:80]}'"
    return True, None


TERMINOLOGY_MAP = {
    "TPT": ["tpt", "tingkat pengangguran terbuka"],
    "IPM": ["ipm", "indeks pembangunan manusia"],
    "PDRB": ["pdrb", "produk domestik regional bruto"],
    "Susenas": ["susenas", "survei sosial ekonomi nasional"],
    "garis kemiskinan": ["garis kemiskinan"],
    "TPAK": ["tpak", "tingkat partisipasi angkatan kerja"],
    "AHH": ["ahh", "angka harapan hidup"],
    "HLS": ["hls", "harapan lama sekolah"],
    "RLS": ["rls", "rata-rata lama sekolah"],
}

def detect_terminology_mismatch(r):
    """Deteksi istilah teknis BPS di response yang TIDAK punya dukungan (bentuk apapun) di context_joined."""
    context_joined = r.get("context_joined", "").lower()
    response = r.get("response", "").lower()

    mismatches = []
    for canonical_key, variants in TERMINOLOGY_MAP.items():
        ctx_has_term = any(
            re.search(r'\b' + re.escape(v) + r'\b', context_joined) for v in variants
        )
        resp_has_term = any(
            re.search(r'\b' + re.escape(v) + r'\b', response) for v in variants
        )
        if resp_has_term and not ctx_has_term:
            mismatches.append({
                "term": canonical_key,
                "issue": "response memakai istilah yang tidak muncul (bentuk apapun) di context",
            })
    return mismatches


def check_terminology_consistency(r):
    """Wrapper boolean untuk detect_terminology_mismatch, dipakai sebagai soft-flag di apply_hard_gates."""
    mismatches = detect_terminology_mismatch(r)
    if mismatches:
        return False, mismatches
    return True, None


def check_degenerate(r):
    """
    Deteksi response kosong/terlalu pendek untuk dianggap tidak informatif.
    Diperbaiki: ambang kata seragam (6) salah menghukum subtask yang secara
    alami ringkas -- source_navigation ('Tabel tersebut adalah Tabel 4.33.'
    hanya 5 kata tapi sudah lengkap dan benar). Ambang sekarang dibaca per
    subtask_type dari CONFIG['degenerate_min_words_by_subtask'].
    """
    resp = (r.get("response") or "").strip()
    subtask = r.get("subtask_type", "")
    min_words = CONFIG["degenerate_min_words_by_subtask"].get(
        subtask, CONFIG["degenerate_min_words"]
    )
    if not resp or len(resp.split()) < min_words:
        return False, "response kosong/terlalu pendek"
    return True, None


def check_table_ref(r):
    """
    Deteksi instruction yang meminta pembacaan ISI SEL TABEL secara
    langsung -- bukan pertanyaan definisi istilah yang kebetulan istilah
    itu muncul sebagai judul kolom tabel. Diperbaiki: versi lama menangkap
    frasa 'dalam tabel ini' secara membabi-buta, padahal instruksi seperti
    'apa yang dimaksud dengan PBI dalam tabel ini' itu sah (bertanya makna
    istilah, bukan minta nilai baris/kolom). Sekarang hard-gate ketat
    hanya untuk pola eksplisit pembacaan sel; frasa generik dikecualikan
    untuk subtask definisi/metodologi yang instruksinya berpola pertanyaan
    definisi (DEFINITION_QUESTION_PATTERNS).
    """
    instr = (r.get("instruction") or "").lower()
    subtask = r.get("subtask_type", "")

    for pat in TABLE_CELL_READ_PATTERNS:
        if re.search(pat, instr, re.IGNORECASE):
            return False, f"instruction meminta pembacaan sel tabel langsung: '{pat}'"

    is_definition_style = any(
        re.search(p, instr, re.IGNORECASE) for p in DEFINITION_QUESTION_PATTERNS
    )
    if subtask in DEFINITION_SUBTASKS and is_definition_style:
        return True, None

    for pat in TABLE_REF_PHRASE_PATTERNS:
        if re.search(pat, instr, re.IGNORECASE):
            return False, f"instruction merujuk tabel langsung: '{pat}'"
    return True, None


def apply_hard_gates(records):
    """
    Menjalankan seluruh pemeriksaan hard-gate dan soft-flag untuk tiap
    record, mengisi dict r['flags']. Urutan: kualitas QA -> kualitas
    context -> bahasa campur (instruction+response) -> skema -> chunk_id
    -> degenerate -> table_ref -> table_residue -> terminology -> cot.
    """
    for r in records:
        context_spans = r.get("context", [])
        context_joined = r.get("context_joined") or "\n\n---\n\n".join(
            context_spans if isinstance(context_spans, list) else [str(context_spans)]
        )
        r["context_joined"] = context_joined

        ok, reason = check_qa_quality(r.get("instruction", ""), r.get("response", ""))
        if not ok:
            r["flags"][reason] = True
            continue

        ok, reason = check_context_quality(context_joined)
        if not ok:
            r["flags"][reason] = True
            continue

        ok, reason = check_mixed_english(r.get("instruction",""), r.get("response",""))
        if not ok:
            r["flags"][reason] = True
            continue

        ok, reason = check_schema(r)
        if not ok:
            r["flags"]["schema_invalid"] = reason

        ok, reason, missing = check_chunk_ids(r)
        if missing:
            r["flags"]["chunk_id_partial_invalid"] = True
            r["flags"]["missing_chunk_ids"] = missing
        if not ok:
            r["flags"]["chunk_id_invalid"] = missing

        ok, reason = check_degenerate(r)
        if not ok:
            r["flags"]["degenerate"] = reason

        ok, reason = check_table_ref(r)
        if not ok:
            r["flags"]["table_ref_instruction"] = reason

        ok, reason = check_table_residue(r.get("context", []))
        if not ok:
            r["flags"]["table_residue"] = reason

        ok, mismatches = check_terminology_consistency(r)
        if not ok:
            r["flags"]["terminology_mismatch"] = mismatches

        ok, reason = check_cot_grounding(r)
        if not ok:
            r["flags"]["cot_ungrounded"] = reason

    return records


NUM_PATTERN = re.compile(r"\d{1,3}(?:\.\d{3})+,\d+|\d{1,3}(?:\.\d{3})+(?!\d)|\d+,\d+|\d+\.\d+(?!\d)|\b\d{2,}\b")

SCALE_PHRASE_PATTERN = re.compile(
    r"(skala|rentang|kisaran|indeks)\b.{0,20}?\b(\d+)\s*(hingga|sampai|s\.?d\.?|-)\s*(\d+)",
    re.IGNORECASE,
)

def normalize_number(n):
    """Normalisasi format angka Indonesia (titik ribuan, koma desimal) ke float untuk perbandingan."""
    if "," in n and "." in n:
        n = n.replace(".", "").replace(",", ".")
    elif "," in n:
        n = n.replace(",", ".")
    elif "." in n:
        integer_part, _, frac_part = n.partition(".")
        if len(frac_part) <= 2:
            pass
        else:
            n = n.replace(".", "")
    try:
        return round(float(n), 2)
    except ValueError:
        return None

def extract_scale_bounds(text):
    """Ekstrak batas bawah/atas skala/rentang yang disebut eksplisit dalam response (bukan angka temuan)."""
    bounds = set()
    for m in SCALE_PHRASE_PATTERN.finditer(text):
        lo, hi = m.group(2), m.group(4)
        bounds.add(normalize_number(lo))
        bounds.add(normalize_number(hi))
    bounds.discard(None)
    return bounds

def compute_numeric_score(r):
    """Hitung proporsi angka dalam response yang punya dukungan eksplisit di context_joined."""
    response = r.get("response", "")
    nums = NUM_PATTERN.findall(response)
    if not nums:
        return 1.0, []

    ctx = r.get("context_joined", "")
    ctx_nums_raw = NUM_PATTERN.findall(ctx)
    ctx_nums_norm = {normalize_number(n) for n in ctx_nums_raw}
    ctx_nums_norm.discard(None)

    scale_bounds = extract_scale_bounds(response)

    unmatched = []
    for n in nums:
        if n in ctx:
            continue
        n_norm = normalize_number(n)
        if n_norm is not None and (n_norm in ctx_nums_norm or n_norm in scale_bounds):
            continue
        unmatched.append(n)

    score = 1.0 - (len(unmatched) / len(nums))
    return score, unmatched

def compute_leaked_phrases(r):
    """
    Deteksi kebocoran meta-talk RAG (rujukan eksplisit ke wadah retrieval
    abstrak). Diperbaiki: sebelumnya merujuk LEAK_HIGH_CONFIDENCE yang
    menangkap 'dokumen ini' secara polos -- padahal itu sering dipakai
    natural untuk merujuk publikasi itu sendiri (mis. 'Gambar 2.1 pada
    dokumen ini'), bukan meta-talk tentang mekanisme RAG. Sekarang
    memakai LEAK_META_PATTERNS (dari Cell 4) yang hanya menangkap
    kombinasi eksplisit menyebut wadah retrieval abstrak.
    """
    text = f"{r.get('instruction','')} {r.get('response','')}".lower()
    return [p for p in LEAK_META_PATTERNS if re.search(p, text, re.IGNORECASE)]

def compute_soft_scores(records):
    """Hitung skor komposit kualitas (track_a_score) dipakai sebagai basis prioritas saat cluster dipangkas."""
    for r in records:
        num_score, unmatched = compute_numeric_score(r)
        leaked = compute_leaked_phrases(r)
        chunk_penalty = 0.2 if r["flags"].get("chunk_id_partial_invalid") else 0.0
        leak_penalty  = min(len(leaked) * 0.3, 1.0)
        cot_penalty   = 0.15 if r["flags"].get("cot_ungrounded") else 0.0

        r["numeric_score"]              = round(num_score, 4)
        r["flags"]["unmatched_numbers"] = unmatched
        if leaked:
            r["flags"]["leaked_phrase"] = leaked
        r["track_a_score"] = round(
            num_score * 0.45
            + (1.0 - leak_penalty) * 0.35
            + (1.0 - chunk_penalty) * 0.1
            + (1.0 - cot_penalty) * 0.1,
            4,
        )
    return records

def flag_near_duplicates_greedy(records, embeddings, threshold, max_per_cluster, config):
    """
    Deteksi cluster near-duplicate via FAISS IndexFlatIP + Union-Find.
    Diperbaiki (iterasi 2): sebelumnya cap longgar hanya berlaku jika
    SELURUH anggota cluster bersubtask definisi istilah -- begitu satu
    anggota cluster kebetulan subtask lain (mis. causal_reasoning),
    seluruh cluster jatuh ke ambang ketat meski ada anggota definisi
    istilah di dalamnya. Sekarang cap dihitung PER-RECORD: anggota
    cluster yang bersubtask definisi istilah dihitung terhadap kuota
    longgar sendiri, anggota subtask lain dihitung terhadap kuota ketat
    sendiri -- keduanya independen dalam satu cluster yang sama.
    """
    n = len(records)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    k = min(20, n)
    scores, ids = index.search(embeddings, k)

    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        px, py = find(x), find(y)
        if px != py:
            parent[py] = px

    for i in range(n):
        for rank in range(1, k):
            j = int(ids[i][rank])
            sim = float(scores[i][rank])
            if sim < threshold:
                break
            union(i, j)

    clusters = defaultdict(list)
    for i in range(n):
        clusters[find(i)].append(i)

    to_remove = set()
    for root, members in clusters.items():
        lenient_members = [
            i for i in members
            if records[i].get("subtask_type") in config["dedup_lenient_subtasks"]
        ]
        strict_members = [i for i in members if i not in lenient_members]

        for group, cap in [
            (lenient_members, config["dedup_max_per_cluster_lenient"]),
            (strict_members, max_per_cluster),
        ]:
            if len(group) <= cap:
                continue
            group_sorted = sorted(
                group,
                key=lambda i: (records[i].get("track_a_score", 0.0), records[i].get("record_id", "")),
                reverse=True,
            )
            for i in group_sorted[cap:]:
                to_remove.add(i)

    for i in to_remove:
        r = records[i]
        best_j = int(ids[i][1])
        r["flags"]["near_duplicate"] = True
        r["flags"]["duplicate_of"]   = records[best_j]["record_id"]
        r["flags"]["similarity"]      = round(float(scores[i][1]), 4)

    return records

### f. Run Fungsi Utama

In [10]:
# 1. Terapkan pembersihan dasar dan Hard Gates (termasuk OCR & Bilingual)
raw_records = apply_hard_gates(raw_records)

# 2. HITUNG SOFT SCORES (Agar nilai track_a_score terisi)
raw_records = compute_soft_scores(raw_records)
avg_score = sum(r["track_a_score"] for r in raw_records) / len(raw_records)
print(f"avg track_a_score: {avg_score:.4f}")

# 3. BUAT EMBEDDING
embed_model = SentenceTransformer(
    CONFIG["embedding_model"],
    trust_remote_code=True,
    device=DEVICE,
)

texts = [f"search_document: {r['instruction']} {r['response']}" for r in raw_records]
embeddings = embed_model.encode(
    texts,
    batch_size=CONFIG["embed_batch_size"],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float32)
print(f"Embeddings shape: {embeddings.shape}")

# 4. LAKUKAN DEDUPLIKASI (sekarang cluster definisi istilah pakai cap longgar)
random.seed(CONFIG["dedup_seed"])
raw_records = flag_near_duplicates_greedy(
    raw_records, embeddings, CONFIG["dedup_threshold"], CONFIG["dedup_max_per_cluster"], CONFIG
)
n_dup = sum(1 for r in raw_records if r["flags"].get("near_duplicate"))
print(f"near_duplicate (greedy): {n_dup}")

avg track_a_score: 0.9518


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Batches:   0%|          | 0/96 [00:00<?, ?it/s]

Embeddings shape: (12261, 768)
near_duplicate (greedy): 232


### g. PREVIEW per kategori flag

In [11]:
def preview_flagged(records, flag_name, reason_fn, n=CONFIG["preview_n"], seed=CONFIG["preview_seed"]):
    flagged = [r for r in records if r["flags"].get(flag_name)]
    print(f"\n{'='*60}\n{flag_name.upper()} — {len(flagged)} records ({len(flagged)/len(records)*100:.1f}%)\n{'='*60}")
    if not flagged:
        return
    sample = random.Random(seed).sample(flagged, min(n, len(flagged)))
    for r in sample:
        print(f"\nid={r['record_id']} | subtask={r.get('subtask_type')}")
        print(f"Q: {r.get('instruction','')[:200]}")
        print(f"A: {r.get('response','')[:200]}")
        print(f"CoT: {r.get('cot','')[:200]}")
        print(f"Alasan: {reason_fn(r)}")
 
preview_flagged(raw_records, "schema_invalid",
                lambda r: r["flags"]["schema_invalid"])
preview_flagged(raw_records, "chunk_id_invalid",
                lambda r: f"≥50% chunk_id invalid: {r['flags']['missing_chunk_ids'][:3]}")
preview_flagged(raw_records, "chunk_id_partial_invalid",
                lambda r: f"sebagian invalid (soft penalti): {r['flags']['missing_chunk_ids'][:3]}")
preview_flagged(raw_records, "degenerate",
                lambda r: r["flags"]["degenerate"])
preview_flagged(raw_records, "table_ref_instruction",
                lambda r: r["flags"]["table_ref_instruction"])
preview_flagged(raw_records, "near_duplicate",
                lambda r: f"mirip dengan {r['flags']['duplicate_of']} (cos={r['flags']['similarity']:.3f})")
preview_flagged(raw_records, "leaked_phrase",
                lambda r: f"frasa bocor: {r['flags']['leaked_phrase']}")
preview_flagged(raw_records, "meta_ref_cleaned",
                lambda r: "meta-reference awalan sudah di-strip")
preview_flagged(raw_records, "high_ocr_noise",
                lambda r: "Konteks mengandung >10% karakter aneh (Sampah OCR tabel)")
preview_flagged(raw_records, "mixed_english_detected",
                lambda r: "Konteks terdeteksi mengandung >15% stopwords Bahasa Inggris (Bilingual)")
preview_flagged(raw_records, "cot_ungrounded",
                lambda r: r["flags"].get("cot_ungrounded", "kutipan cot tidak ditemukan verbatim di context"))


SCHEMA_INVALID — 0 records (0.0%)

CHUNK_ID_INVALID — 0 records (0.0%)

CHUNK_ID_PARTIAL_INVALID — 0 records (0.0%)

DEGENERATE — 0 records (0.0%)

TABLE_REF_INSTRUCTION — 40 records (0.3%)

id=709cc617af409322 | subtask=methodology_explanation
Q: Bagaimana cara menafsirkan estimasi yang memiliki RSE di atas 50 persen menurut catatan dalam tabel ini?
A: Menurut catatan pada tabel, jika nilai RSE lebih dari 50 persen, estimasi tersebut dianggap tidak akurat. Oleh karena itu, estimasi dengan RSE di atas 50 persen sebaiknya tidak digunakan untuk keperlu
CoT: Teks menyatakan bahwa jika RSE >50%, estimasi dianggap tidak akurat. Karena itu, estimasi dengan RSE di atas 50 persen tidak boleh digunakan sebagai angka yang andal.
Alasan: instruction merujuk tabel langsung: 'dalam tabel ini'

id=2f820214a8db01d4 | subtask=methodology_explanation
Q: Bagaimana metode pengukuran yang digunakan untuk menghasilkan angka 2,11 persen dan 3,80 persen pada baris Indonesia dalam tabel ini?
A: Metode penguk

### h. DISTRIBUSI track_a_score per subtask

In [12]:
by_stratum = defaultdict(list)
for r in raw_records:
    by_stratum[r.get("subtask_type")].append(r["track_a_score"])
 
print(f"\n{'Subtask':<35}{'N':>7}{'Mean':>8}{'Min':>8}")
for sub, scores in sorted(by_stratum.items()):
    print(f"{sub:<35}{len(scores):>7}{np.mean(scores):>8.3f}{np.min(scores):>8.3f}")


Subtask                                  N    Mean     Min
causal_reasoning                      1337   0.950   0.550
comparative_analysis                  1453   0.929   0.535
definition_clarification              1492   0.969   0.535
document_overview                      763   0.947   0.535
factual_retrieval                     1778   0.948   0.535
methodology_explanation               1444   0.966   0.535
multi_source_synthesis                1308   0.943   0.550
source_navigation                     1357   0.966   0.535
trend_interpretation                  1329   0.946   0.550


### i. FINALISASI — hard gate + export

In [13]:
HARD_GATE_KEYS = [
    "high_ocr_noise",
    "mixed_english_detected",
    "schema_invalid",
    "chunk_id_invalid",
    "degenerate",
    "table_ref_instruction",
    "near_duplicate",
    "table_residue",
]
 
filtered_records = [r for r in raw_records if not any(k in r["flags"] for k in HARD_GATE_KEYS)]
removed_records  = [r for r in raw_records if any(k in r["flags"] for k in HARD_GATE_KEYS)]
 
print(f"\nTotal raw       : {len(raw_records)}")
print(f"Lolos hard gate : {len(filtered_records)} ({len(filtered_records)/len(raw_records)*100:.1f}%)")
print(f"Gagal hard gate : {len(removed_records)} ({len(removed_records)/len(raw_records)*100:.1f}%)")
 
n_cot_ungrounded = sum(1 for r in filtered_records if r["flags"].get("cot_ungrounded"))
print(f"Lolos tapi cot_ungrounded (soft flag, tetap masuk filtered): {n_cot_ungrounded}")
 
OUT = CONFIG["output_dir"]
FILTERED_PATH = f"{OUT}/filtered_records.jsonl"
REMOVED_PATH  = f"{OUT}/removed_records.jsonl"
REPORT_PATH   = f"{OUT}/qc_tahap1_report.json"
 
with open(FILTERED_PATH, "w", encoding="utf-8") as f:
    for r in filtered_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
 
with open(REMOVED_PATH, "w", encoding="utf-8") as f:
    for r in removed_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
 
removal_reasons = defaultdict(int)
for r in removed_records:
    for k in HARD_GATE_KEYS:
        if k in r["flags"]:
            removal_reasons[k] += 1
 
report = {
    "timestamp":    datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "total_raw":    len(raw_records),
    "total_filtered": len(filtered_records),
    "total_removed":  len(removed_records),
    "n_meta_ref_cleaned": n_cleaned,
    "n_cot_ungrounded_soft": n_cot_ungrounded,
    "removal_reasons": dict(removal_reasons),
    "track_a_score_by_subtask": {
        s: {"n": len(v), "mean": float(np.mean(v)), "min": float(np.min(v))}
        for s, v in by_stratum.items()
    },
    "config": {
        k: (list(v) if isinstance(v, set) else v)
        for k, v in CONFIG.items() if k != "skip_section_sources"
    },
}
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
 
print(f"\nRemoval reasons:")
for k, v in removal_reasons.items():
    print(f"  {k:<25}: {v}")
print(f"\nOutput:")
print(f"  {FILTERED_PATH}")
print(f"  {REMOVED_PATH}")
print(f"  {REPORT_PATH}")


Total raw       : 12261
Lolos hard gate : 9350 (76.3%)
Gagal hard gate : 2911 (23.7%)
Lolos tapi cot_ungrounded (soft flag, tetap masuk filtered): 205

Removal reasons:
  table_residue            : 1896
  table_ref_instruction    : 40
  near_duplicate           : 232
  high_ocr_noise           : 749
  mixed_english_detected   : 80

Output:
  output_05b/filtered_records.jsonl
  output_05b/removed_records.jsonl
  output_05b/qc_tahap1_report.json
